# Hansen Ch.4 习题解答（计算部分）

理论见 `Hansen_Ch04_Exercises_Solutions.md`。

本 notebook：Exercise **4.24–4.26**。

## 公共函数：OLS + HC0–HC3

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

def ols_hc(y, X):
    n, k = X.shape
    beta = np.linalg.lstsq(X, y, rcond=None)[0]
    e = y - X @ beta
    XXinv = np.linalg.inv(X.T @ X)
    h = np.sum(X * (X @ XXinv), axis=1)
    def sand(scale):
        u = X * (scale * e)[:, None]
        return XXinv @ (u.T @ u) @ XXinv
    Vhom = XXinv * (np.sum(e**2) / (n - k))
    V0 = sand(np.ones(n))
    V1 = V0 * (n / (n - k))
    V2 = sand(1.0 / np.sqrt(np.clip(1 - h, 1e-12, None)))
    V3 = sand(1.0 / np.clip(1 - h, 1e-12, None))
    se = lambda V: np.sqrt(np.diag(V))
    return dict(beta=beta, e=e, n=n, k=k,
                se_hom=se(Vhom), se_HC0=se(V0), se_HC1=se(V1),
                se_HC2=se(V2), se_HC3=se(V3), V_HC3=V3)

CPS = Path("../../hansen/econometrics/data/cps09mar/cps09mar.xlsx")
if not CPS.exists():
    CPS = Path("/home/fang/Project/zhihu-paper/p1/hansen/econometrics/data/cps09mar/cps09mar.xlsx")
df = pd.read_excel(CPS)
df["experience"] = df["age"] - df["education"] - 6
df["lwage"] = np.log(df["earnings"] / (df["hours"] * df["week"]))
df["exp2"] = (df["experience"] ** 2) / 100
print("CPS n=", len(df))


## Exercise 4.24：方程 (3.49) 多种 SE

In [ ]:
mask = (df.race==4)&(df.marital==7)&(df.female==0)&(df.experience<45)
s = df.loc[mask]
y = s.lwage.to_numpy(float)
X = np.c_[s.education, s.experience, s.exp2, np.ones(len(s))]
names = ["education","experience","exp2/100","intercept"]
r = ols_hc(y, X)
tab = pd.DataFrame({
    "beta": r["beta"], "SE_hom": r["se_hom"], "HC0": r["se_HC0"],
    "HC1": r["se_HC1"], "HC2": r["se_HC2"], "HC3": r["se_HC3"],
}, index=names)
print("n=", r["n"])
tab


## Exercise 4.25：白人男性西班牙裔，HC3

In [ ]:
m = (df.race==1)&(df.female==0)&(df.hisp==1)
s = df.loc[m].copy()
s["married"] = s.marital.isin([1,2,3]).astype(float)
s["wid_div"] = s.marital.isin([4,5]).astype(float)
s["separated"] = (s.marital==6).astype(float)
s["NE"] = (s.region==1).astype(float)
s["South"] = (s.region==3).astype(float)
s["West"] = (s.region==4).astype(float)
y = s.lwage.to_numpy(float)
X = np.c_[s.education,s.experience,s.exp2,s.NE,s.South,s.West,s.married,s.wid_div,s.separated,np.ones(len(s))]
names = ["education","experience","exp2/100","NE","South","West","married","wid_div","separated","intercept"]
r = ols_hc(y, X)
pd.DataFrame({"beta": r["beta"], "HC3": r["se_HC3"]}, index=names)


## Exercise 4.26：DDK2011 tracking + 聚类 SE

In [ ]:
DDK = Path("../../hansen/econometrics/data/DDK2011/DDK2011.xlsx")
if not DDK.exists():
    DDK = Path("/home/fang/Project/zhihu-paper/p1/hansen/econometrics/data/DDK2011/DDK2011.xlsx")
ddk = pd.read_excel(DDK)
for c in ddk.columns:
    ddk[c] = pd.to_numeric(ddk[c], errors="coerce")
ts = ddk["totalscore"]
ddk["ystd"] = (ts - ts.mean()) / ts.std()

# baseline: tracking only
d0 = ddk[["ystd","tracking"]].dropna()
X0 = np.c_[d0.tracking.values, np.ones(len(d0))]
b0 = np.linalg.lstsq(X0, d0.ystd.values, rcond=None)[0]
print("tracking only (n=%d): intercept=%.3f, tracking=%.3f" % (len(d0), b0[1], b0[0]))

d = ddk[["ystd","tracking","agetest","girl","etpteacher","percentile","schoolid"]].dropna()
y = d.ystd.values
X = np.c_[d.tracking, d.agetest, d.girl, d.etpteacher, d.percentile, np.ones(len(d))]
names = ["tracking","age","girl","etpteacher","percentile","intercept"]
beta = np.linalg.lstsq(X, y, rcond=None)[0]
e = y - X @ beta
XXinv = np.linalg.inv(X.T @ X)
u = X * e[:, None]
Vhc = XXinv @ (u.T @ u) @ XXinv
meat = np.zeros((X.shape[1], X.shape[1]))
for g in np.unique(d.schoolid.values):
    idx = d.schoolid.values == g
    sc = X[idx].T @ e[idx]
    meat += np.outer(sc, sc)
Vcl = XXinv @ meat @ XXinv
out = pd.DataFrame({
    "beta": beta,
    "SE_robust": np.sqrt(np.diag(Vhc)),
    "SE_cluster": np.sqrt(np.diag(Vcl)),
}, index=names)
out["ratio"] = out["SE_cluster"] / out["SE_robust"]
print("n=", len(d), "schools=", d.schoolid.nunique())
out
